# Skin Risk Dataset V4 — Colab pipeline

Run each cell in order. The original dataset is never overwritten.

After `review`, inspect `review.csv` before `rebuild` if you can. `CHECK_LABEL` is never auto-relabelled.

In [ ]:
# @title Paths
from google.colab import drive
drive.mount("/content/drive")

DATASET_ROOT = "/content/drive/MyDrive/your_original_dataset"  # folder that contains train/ val/ test/
OUTPUT_ROOT = "/content/pipeline_output"
REPO_DIR = "/content/vinu"
SEED = 42

In [ ]:
# Upload this repo (or clone it), then install pipeline dependencies.
import os, sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!pip install -q pillow numpy pandas imagehash opencv-python-headless scikit-learn tqdm PyYAML

os.environ["DATASET_ROOT"] = DATASET_ROOT
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

## 1. Validate

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED validate

## 2. Exact duplicates

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED exact-dups

## 3. Near-duplicates (leakage-critical)

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED near-dups

## 4. Quality

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED quality

## 5. Risk-label consistency (within lesion type only)

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED labels

## 6. Manual review set

Open `OUTPUT_ROOT/reports/review.csv` and `OUTPUT_ROOT/Review/`.
Set `decision` to `KEEP` only when you accept the current label.

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED review

## 7–9. Rebuild, statistics, leakage check

In [ ]:
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED rebuild
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED stats
!python run_pipeline.py --dataset-root "$DATASET_ROOT" --output-root "$OUTPUT_ROOT" --seed $SEED leakage-check

## Train on V4 only

Runtime → GPU. Test is evaluated once at the end. Training accuracy is not test accuracy.

In [ ]:
V4 = f"{OUTPUT_ROOT}/Skin_Risk_Dataset_V4"
!python train_v4.py --data-root "$V4" --output-dir /content/training_output --epochs 20 --batch-size 16 --seed 42